# 00_vannverk datasett

In [36]:
import pandas as pd

In [37]:
df = pd.read_csv("../data/processed/00_vannverk.csv", encoding="utf-8-sig")

print("Shape (rader, kolonner):", df.shape)

print("\nAntall duplikate rader (mtid_vf + periode):",
      df.duplicated(["mtid_vf", "periode"]).sum())

print("\nAntall anlegg per år:")
print(df.periode.value_counts().sort_index())

print("\nAndel manglende verdier per kolonne (topp 20):")
print(df.isna().mean().sort_values(ascending=False).head(20))

print("\nOppsummeringsstatistikk for total_analyser og total_avvik:")
print(df[["total_analyser", "total_avvik"]].describe())

Shape (rader, kolonner): (36034, 80)

Antall duplikate rader (mtid_vf + periode): 0

Antall anlegg per år:
periode
2008     412
2009    2004
2010    2298
2011    2287
2012    2277
2013    2315
2014    2321
2015    2142
2016    2262
2017    2178
2018    2022
2019    1998
2020    2017
2021    1981
2022    1879
2023    1899
2024    1903
2025    1839
Name: count, dtype: int64

Andel manglende verdier per kolonne (topp 20):
noedvann_inngaar_alt_kilde    0.624688
max_vann_dogn                 0.517900
max_vann_pers                 0.506244
forbr_annet                   0.472748
forbr_industri                0.453877
forbr_tjytende                0.453044
forbr_primnaering             0.446745
forbr_fritidsboliger          0.441416
forbr_lekkasje                0.422101
forbr_fast_bosetting          0.406200
ledn_gjsn                     0.211828
ledn_avvik                    0.211328
ledn_analyser                 0.195454
vann_mottatt                  0.177555
enterok_gjsn                  0

In [38]:
d = df[df.total_analyser > 0].copy()
print("Antall vannverk med minst én analyse:", len(d))
print("Andel av disse med minst ett avvik:", (d.total_avvik > 0).mean())

cols = ["vannprod", "ant_fastboende", "forbr_fast_bosetting",
        "max_vann_dogn", "noedvann_inngaar_alt_kilde"]
print("\nAndel manglende verdier per år, for utvalgte kolonner:")
print(df.groupby("periode")[cols].apply(lambda x: x.isna().mean()).round(2))
d = df[(df.total_analyser > 0) & df.total_avvik.notna() & (df.periode >= 2009)].copy()
d["avvik"] = (d.total_avvik > 0).astype(int)
print(len(d), d.avvik.mean())

Antall vannverk med minst én analyse: 34307
Andel av disse med minst ett avvik: 0.5662692744920862

Andel manglende verdier per år, for utvalgte kolonner:
         vannprod  ant_fastboende  forbr_fast_bosetting  max_vann_dogn  \
periode                                                                  
2008          0.0            0.01                  0.98           1.00   
2009          0.0            0.01                  0.53           1.00   
2010          0.0            0.00                  0.52           1.00   
2011          0.0            0.00                  0.50           1.00   
2012          0.0            0.00                  0.48           1.00   
2013          0.0            0.00                  0.47           0.71   
2014          0.0            0.00                  0.48           0.68   
2015          0.0            0.00                  0.38           0.57   
2016          0.0            0.00                  0.39           0.48   
2017          0.0            0.

In [39]:
d = df[(df.total_analyser > 0) & df.total_avvik.notna() & (df.periode >= 2009)].copy()
d["avvik"] = (d.total_avvik > 0).astype(int)

print("Antall anlegg etter filtrering (analyser>0, avvik kjent, periode>=2009):", len(d))
print("Andel med minst ett avvik:", d.avvik.mean())

Antall anlegg etter filtrering (analyser>0, avvik kjent, periode>=2009): 33752
Andel med minst ett avvik: 0.57531405546338


In [40]:
oversikt = pd.DataFrame({
    "type": df.dtypes,
    "andel_mangler": df.isna().mean().round(2),
    "antall_unike": df.nunique(),
    "vanligste_verdi_andel": df.apply(lambda s: s.value_counts(normalize=True, dropna=True).iloc[0] if s.notna().any() else None).round(2),
})

print("Kolonneoversikt (type, andel manglende, antall unike, andel for vanligste verdi), sortert etter andel manglende:")
print(oversikt.sort_values("andel_mangler").to_string())

Kolonneoversikt (type, andel manglende, antall unike, andel for vanligste verdi), sortert etter andel manglende:
                               type  andel_mangler  antall_unike  vanligste_verdi_andel
mtid_vf                      object           0.00          4053                   0.00
ant_hytter                  float64           0.00           299                   0.32
ant_husstander              float64           0.00           696                   0.14
ant_personer_max            float64           0.00           736                   0.05
ant_fastboende              float64           0.00           783                   0.16
beredsk_oppd                 object           0.00             2                   0.54
beredsk_ovelse               object           0.00             2                   0.68
vannuttak                   float64           0.00         18608                   0.06
aktiv                        object           0.00             2                   0.76
navn   

In [41]:
rens = ["antall_anlegg", "uv", "klorering", "koagulering",
        "membranfiltrering", "siling", "lufting", "ph_justering"]

print("Andel manglende verdier per rensemetode-kolonne:")
print(df[rens].isna().mean().round(3))

print("\nAndel rader der ALLE rensemetode-kolonnene mangler samtidig:")
print(df[rens].isna().all(axis=1).mean().round(3))

print("\nOrganisasjonsformer med flest rader der 'uv' mangler (topp 5):")
print((df[df.uv.isna()].groupby("orgform").size() / df.groupby("orgform").size()).sort_values(ascending=False).head(10))

Andel manglende verdier per rensemetode-kolonne:
antall_anlegg        0.125
uv                   0.125
klorering            0.125
koagulering          0.125
membranfiltrering    0.125
siling               0.125
lufting              0.125
ph_justering         0.125
dtype: float64

Andel rader der ALLE rensemetode-kolonnene mangler samtidig:
0.125

Organisasjonsformer med flest rader der 'uv' mangler (topp 5):
orgform
TVAM    1.000000
OPMV    1.000000
ANNA    0.888889
SAM     0.406593
ORGL    0.285714
ADOS    0.272727
ANS     0.270000
ENK     0.268065
FLI     0.260087
BEDR    0.187866
dtype: float64


**Datakvalitet og innledende funn**

Datasettet består av 36 034 rader (anlegg × år) og 80 kolonner, med dekning fra 2008 til 2025. Nøkkelen mtid_vf + periode er unik (0 duplikater).

Responsvariabel (avvik): Etter å filtrere til anlegg med minst én analyse, kjent avvik-status og periode ≥ 2009, sitter vi igjen med 33 752 rader. Andelen anlegg med minst ett avvik ligger stabilt på ca. 56-58 %, altså en godt balansert klasse — gunstig utgangspunkt for klassifikasjon.

Missing data er strukturell, ikke tilfeldig:

noedvann_inngaar_alt_kilde er 100 % manglende før 2019 og 0 % manglende fra 2019 og utover. Variabelen ble tydeligvis innført i rapporteringsskjemaet det året, og bør enten ekskluderes for tidligere år eller kun brukes i en modell begrenset til 2019+.
max_vann_dogn og forbr_fast_bosetting viser samme mønster i mildere form: andelen manglende faller jevnt fra rundt 50-98 % i tidlige år til under 10-30 % i nyere år, som tyder på gradvis forbedret rapporteringspraksis over tid.
Rensemetode-kolonnene (uv, klorering, koagulering, membranfiltrering, siling, lufting, ph_justering, antall_anlegg) mangler alle samtidig i nøyaktig 12,5 % av radene — dette er ikke spredt tilfeldig missing, men anlegg som ikke har fylt ut denne delen av skjemaet i det hele tatt.
Manglende rensemetode-data henger tydelig sammen med organisasjonsform (orgform): organisasjonstyper som TVAM og OPMV mangler nesten 100 %, mens KOMM (kommunalt) og BEDR (bedrift) har klart best rapportering (under 20-25 % manglende). Dette peker mot at missing sannsynligvis reflekterer manglende rapportering fra mindre/uvanlige anleggstyper, ikke faktisk fravær av rensing.

# 01_vannverk datasett

In [45]:
df = pd.read_csv("../data/processed/01_vannverk.csv", encoding="utf-8-sig")

print("Gjennomsnitt av bakterie-avvik-kolonner, gruppert på avvik:")
print(df.groupby("avvik")[["koli_avvik", "ecoli_avvik", "enterok_avvik"]].mean())

identisk_avvik = (df["avvik"] == (df["total_avvik"] > 0).astype(int)).mean()
print(f"\nAndel rader der avvik == (total_avvik > 0): {identisk_avvik:.3f}")

identisk_rate = ((df["total_avvik"] / df["total_analyser"]).round(6) == df["avvik_rate"].round(6)).mean()
print(f"Andel rader der avvik_rate == total_avvik / total_analyser: {identisk_rate:.3f}")

print("\nKonklusjon: total_avvik, avvik_rate, koli_avvik, ecoli_avvik, enterok_avvik")
print("og bakterie_avvik er direkte avledet av target-variabelen 'avvik' og må")
print("fjernes fra featuresettet før modellering (data leakage).")

leakage_kolonner = [
    "total_avvik", "avvik_rate",
    "koli_avvik", "ecoli_avvik", "enterok_avvik", "bakterie_avvik"
]
features = df.drop(columns=leakage_kolonner + ["avvik", "mtid_vf", "navn"])
target = df["avvik"]

print(f"\nAntall features etter fjerning av leakage-kolonner: {features.shape[1]}")
print(list(features.columns))

Gjennomsnitt av bakterie-avvik-kolonner, gruppert på avvik:
       koli_avvik  ecoli_avvik  enterok_avvik
avvik                                        
0        0.000000     0.000000       0.000000
1        1.538645     0.257321       0.131862

Andel rader der avvik == (total_avvik > 0): 1.000
Andel rader der avvik_rate == total_avvik / total_analyser: 1.000

Konklusjon: total_avvik, avvik_rate, koli_avvik, ecoli_avvik, enterok_avvik
og bakterie_avvik er direkte avledet av target-variabelen 'avvik' og må
fjernes fra featuresettet før modellering (data leakage).

Antall features etter fjerning av leakage-kolonner: 24
['periode', 'kommune', 'total_analyser', 'vannprod', 'ant_fastboende', 'ant_hytter', 'ant_husstander', 'antall_anlegg', 'uv', 'klorering', 'koagulering', 'membranfiltrering', 'siling', 'lufting', 'ph_justering', 'orgform', 'aktiv', 'boliger', 'helseinst', 'skoler', 'hyttercamp', 'gardsbruk', 'beredsk_oppd', 'beredsk_ovelse']
